# Advanced RAG Pipeline : Recherche Hybride et Reranking

Ce notebook implémente la partie **Inférence et Génération** du projet.
Contrairement à une approche RAG classique (Question / Vecteurs / Réponse), nous déployons ici une architecture pour maximiser la précision et réduire les hallucinations.

**Objectif Final :** Comparer en temps réel un RAG Naïf (V1) contre cette architecture Hybride (V4).

---

# 1. Initialisation de l'environnement "Advanced RAG"

Cette première cellule charge une stack technique complète pour un système de Questions-Réponses haute performance.

Nous combinons ici plusieurs approches pour maximiser la pertinence :
1.  **Recherche Hybride** : Mots-clés (`rank_bm25`) + Vecteurs (`faiss`).
2.  **Reranking** : Ré-évaluation précise des résultats (`CrossEncoder`).
3.  **Génération Locale** : LLM tournant en local (`transformers`).

### Détail des bibliothèques :

* **Data et Calcul** : `pandas`, `numpy`, `faiss` (Vector Database), `torch` (PyTorch pour l'accélération GPU).
* **Retrieval (Recherche)** :
    * `sentence_transformers` : Pour créer les embeddings de recherche.
    * `rank_bm25` : Pour la recherche lexicale classique (BM25Okapi), utile pour les mots-clés exacts ou les acronymes.
* **Refinement (Reranking)** :
    * `CrossEncoder` : Un modèle plus lourd mais plus précis qui compare la question et le document côte à côte pour trier les meilleurs résultats.
* **Generation (LLM)** :
    * `AutoTokenizer`, `AutoModelForCausalLM` : Pour charger un modèle génératif (type Mistral, Llama, ou Phi) directement via HuggingFace.

In [7]:
# IMPORTS
from pathlib import Path
import textwrap
import warnings

# Bibliothèques de calcul et data
import faiss
import numpy as np
import pandas as pd
import torch

# Bibliothèques NLP (Sémantique & Reranking)
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

# Bibliothèque NLP (Mots-clés)
from rank_bm25 import BM25Okapi 

import sys
from pathlib import Path

## 2. Configuration Globale et Optimisation

Nous définissons ici les paramètres qui piloteront le comportement du pipeline RAG.

### Le Modèle (LLM)
Nous utilisons **`meta-llama/Llama-3.2-1B-Instruct`**.

### Stratégie de Filtrage (Thresholds)
Pour éviter les hallucinations, nous appliquons un **deux filtres** :



| Paramètre | Valeur | Rôle |
| :--- | :--- | :--- |
| **`THRESHOLD_SIMPLE`** | `0.45` | **Premier filtre (FAISS)** : Élimine les documents trop éloignés sémantiquement (Score Cosinus). |
| **`THRESHOLD_RERANK`** | `0.00` | **Second filtre (Cross-Encoder)** : Le re-ranker sort un score non borné. Une valeur positive (> 0) signifie "Pertinent", une valeur négative signifie "Non pertinent". |

In [2]:
# CONFIGURATION GLOBALE

# Le modèle LLM
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# Seuils de décision
THRESHOLD_SIMPLE = 0.45   # Pour FAISS (0 à 1).
THRESHOLD_RERANK = 0.00   # Pour Cross-Encoder (Logits -10 à +10). < 0 signifie Non pertinent.

print(f"[INIT] Loading LLM model: {MODEL_NAME}...")

# Détection automatique du matériel (GPU ou CPU)
if torch.cuda.is_available() or torch.backends.mps.is_available():
    llm_dtype = torch.bfloat16 # Mode rapide et léger pour GPU
    print("[INIT] Mode: GPU Acceleration (bfloat16)")
else:
    llm_dtype = torch.float32  # Mode standard pour CPU
    print("[INIT] Mode: CPU (float32)")

[INIT] Loading LLM model: meta-llama/Llama-3.2-1B-Instruct...
[INIT] Mode: GPU Acceleration (bfloat16)


## 3. Chargement du LLM

Nous chargeons maintenant les poids du modèle **Llama-3.2-1B-Instruct** en mémoire via la librairie `transformers`.

Cela se fait en deux composants :

1.  **Le Tokenizer** : Il découpe le texte brut en tokens et les convertit en nombres.
2.  **Le Modèle Causal** : C'est le réseau de neurones lui-même qui prédit la suite du texte.

In [4]:
# CHARGEMENT DU LLM

# 1. Le Tokenizer : Convertit le texte en nombres
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. Le Modèle
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=llm_dtype,
    device_map="auto", # Gestion automatique de la mémoire (VRAM/RAM)
)

## 4. Fonctions de Génération et Prompt Engineering

Nous définissons ici les fonctions qui interagissent directement avec le LLM.

### Points clés de l'implémentation :

1.  **Prompt Système Strict (`call_llm`)** :
    Nous injectons une instruction système forte : *"Answer using ONLY the provided contexts."*.
    * **But** : Forcer le modèle à agir comme un analyste factuel et empêcher les "hallucinations" (inventer des faits).
2.  **Génération Déterministe** :
    Nous utilisons le paramètre `do_sample=False`.
    * Le modèle choisit toujours le mot le plus probable. C'est crucial pour un assistant technique où l'on veut de la précision.
3.  **Gestion du Fallback (`_fallback`)** :
    Si le moteur de recherche ne trouve aucun document pertinent, le script bascule automatiquement sur la "mémoire interne" du modèle (`call_llm_baseline`). L'utilisateur est averti que la réponse provient de connaissances générales et non des documents.



### Structure du Prompt RAG (`build_prompt`)
Le modèle reçoit un texte formaté ainsi :
```text
SYSTEM: You are a precise research assistant...
USER:
CONTEXTS:
[Context 1 | score=0.85]
...texte du document...

[Context 2 | score=0.72]
...texte du document...

QUESTION:
How do you make hot chocolate?

Answer based on contexts only.

In [5]:
# FONCTIONS UTILITAIRES LLM

def call_llm(prompt: str) -> str:
    """
    Envoie le prompt au LLM et récupère la réponse générée.
    Gère la mise en forme du template de chat (System + User).
    """
    # Définition du rôle système strict pour limiter les hallucinations
    messages = [
        {"role": "system", "content": "You are a precise research assistant. Answer using ONLY the provided contexts. If the answer is not in the contexts, say you don't know."},
        {"role": "user", "content": prompt},
    ]
    
    # Transformation des messages en tokens
    if hasattr(tokenizer, "apply_chat_template"):
        model_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    else:
        # Fallback manuel si la fonction n'existe pas
        chat_text = f"System: Helper.\nUser:\n{prompt}\n\nAssistant:"
        model_inputs = tokenizer(chat_text, return_tensors="pt", truncation=True, max_length=4096)

    # Si model_inputs est juste un Tensor, on le met dans un dictionnaire
    if isinstance(model_inputs, torch.Tensor):
        model_inputs = {"input_ids": model_inputs}
    
    # Envoi des données sur le bon périphérique (GPU ou CPU)
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items() if isinstance(v, torch.Tensor)}

    # Génération
    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs, 
            max_new_tokens=512, 
            do_sample=False, # Déterministe, pas de créativité aléatoire
            pad_token_id=tokenizer.eos_token_id
        )

    # Décodage (Tokens vers Texte)
    generated_ids = output_ids[0][model_inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def call_llm_baseline(query: str) -> str:
    """Appelle le LLM sans contexte (Mémoire interne uniquement)."""
    messages = [{"role": "user", "content": query}]
    
    model_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    
    # Correction par sécurité
    if isinstance(model_inputs, torch.Tensor):
        model_inputs = {"input_ids": model_inputs}
        
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items() if isinstance(v, torch.Tensor)}

    with torch.no_grad():
        output_ids = model.generate(**model_inputs, max_new_tokens=512, do_sample=False)
    return tokenizer.decode(output_ids[0][model_inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def build_prompt(query: str, contexts):
    """Construit le prompt final avec les documents récupérés."""
    context_str = "\n\n".join([f"[Context {i+1} | score={c.get('score', 0):.2f}]\n{c['text']}" for i, c in enumerate(contexts)])
    return f"CONTEXTS:\n{context_str}\n\nQUESTION:\n{query}\n\nAnswer based on contexts only."

def _fallback(query, info=""):
    """Gère le cas où aucun document pertinent n'est trouvé."""
    baseline = call_llm_baseline(query)
    return f"*** [RAG FAILED: {info}] Fallback to Baseline (Connaissances générales) ***\n\n{baseline}"

## 5. Chargement des Ressources et Indexation Hybride

Cette fonction prépare le terrain pour une **Recherche Hybride (Hybrid Search)**. Contrairement à une recherche vectorielle simple, nous chargeons ici plusieurs outils pour maximiser la précision.



**Les 3 piliers de notre système de recherche :**

1.  **Dense Retrieval (FAISS + Bi-Encoder)** :
    * Comprend le **sens** global et le contexte (ex: "problème d'argent" sera mis en relation avec "déficit financier").
    * Utilise l'index pré-calculé sur le disque.
2.  **Sparse Retrieval (BM25)** :
    * Comprend les **mots-clés exacts** (ex: "Article 42-B", "2024").
    * Nous construisons l'index BM25 à la volée (in-memory) à partir du DataFrame.
3.  **Reranking (Cross-Encoder)** :
    * Utilise le modèle `ms-marco-MiniLM-L-6-v2`.
    * Il interviendra *après* la recherche pour ré-ordonner les résultats en lisant attentivement le couple (Question, Document).

In [8]:
# GESTION DES RESSOURCES (Index, Corpus, Modèles)

sys.path.append("..")
BASE_DIR = Path("..")

PROC_DIR = BASE_DIR / "data" / "processed"
INDEX_DIR = BASE_DIR / "data" / "index"


def load_resources():
    print("--- Loading RAG Resources ---")
    
    # 1. Corpus Textuel (CSV)
    corpus_path = PROC_DIR / "docs_corpus.csv"
    if not corpus_path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {corpus_path}. Lancez docs_to_corpus.py d'abord.")
    df = pd.read_csv(corpus_path)
    
    # 2. Index Vectoriel (FAISS)
    faiss_path = INDEX_DIR / "corpus.index"
    index = faiss.read_index(str(faiss_path))

    # 3. Modèle d'Embedding (Bi-Encoder)
    model_name_path = INDEX_DIR / "embedding_model.txt"
    model_name = model_name_path.read_text(encoding="utf-8").strip()
    embed_model = SentenceTransformer(model_name)
    
    # 4. Modèle de Reranking (Cross-Encoder)
    print("Loading Reranker (Cross-Encoder)...")
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    # 5. Index Mots-clés (BM25)
    print("Building BM25 Index (Sparse Retrieval)...")
    # Tokenisation simple par espace pour BM25
    tokenized_corpus = [str(doc).lower().split(" ") for doc in df['text']]
    bm25 = BM25Okapi(tokenized_corpus)

    return df, index, embed_model, reranker, bm25



## 6. Stratégie de Recherche Hybride & Reranking

C'est le cœur de notre moteur "Advanced RAG". Plutôt que de faire confiance à une seule méthode, nous combinons trois techniques pour ne rien rater.



**Le Workflow de Récupération :**

1.  **Dense Retrieval (`retrieve_faiss`)** :
    * Utilise les vecteurs (embeddings).
    * *Force :* Comprend le sens global, les synonymes et le contexte.
    * *Faiblesse :* Peut rater des acronymes spécifiques ou des noms propres rares.
2.  **Sparse Retrieval (`retrieve_bm25`)** :
    * Utilise les mots-clés exacts (algorithme probabiliste).
    * *Force :* Trouve les correspondances exactes (ex: "Article 12", "Code Error 504").
    * *Faiblesse :* Ne comprend pas le sens (ne sait pas que "voiture" = "automobile").
3.  **Fusion RRF (`reciprocal_rank_fusion`)** :
    * **Problème :** Les scores FAISS (0 à 1) et BM25 (0 à 15+) ne sont pas comparables.
    * **Solution :** On utilise le rang (1er, 2ème, 3ème...) plutôt que le score brut.
    * **Formule :** $Score = \frac{1}{k + rang}$. Si un document est bien classé par les deux méthodes, il remonte en tête.
4.  **Reranking (`rerank_contexts`)** :
    * Les meilleurs candidats de la fusion sont envoyés au **Cross-Encoder**.
    * C'est un modèle plus lent mais très intelligent qui "lit" la question et le document côte à côte pour donner un score de pertinence final ultra-précis.

In [9]:
# BRIQUES DE BASE (Retrieval Modules)

def retrieve_faiss(query: str, df, index, embed_model, top_k=10):
    """Recherche Vectorielle (Dense Retrieval) via FAISS."""
    query_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_emb, top_k)
    contexts = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(df): continue
        row = df.iloc[idx]
        contexts.append({
            "doc_id": row.get("doc_id", idx),
            "score": float(score),
            "text": str(row["text"]),
            "source": "FAISS"
        })
    return contexts

def retrieve_bm25(query: str, df, bm25, top_k=10):
    """Recherche Lexicale (Sparse Retrieval) via BM25."""
    tokenized_query = query.lower().split(" ")
    top_docs = bm25.get_top_n(tokenized_query, df['text'].tolist(), n=top_k)
    
    contexts = []
    for text in top_docs:
        contexts.append({
            "doc_id": "BM25_Match",
            "score": 0.0, # BM25 score non normalisé, ignoré ici au profit du rang
            "text": text,
            "source": "BM25"
        })
    return contexts

def reciprocal_rank_fusion(list_a, list_b, k=60):
    """
    Fusionne deux listes de résultats (ex: FAISS + BM25) via RRF.
    Score = 1 / (k + rang). Favorise les documents présents dans les deux listes.
    """
    scores_map = {}
    
    def add_to_map(results_list):
        for rank, doc in enumerate(results_list):
            key = doc['text'] # Clé de dédoublage
            if key not in scores_map:
                scores_map[key] = {"doc": doc, "score": 0.0}
            scores_map[key]["score"] += 1 / (k + rank + 1)
            
    add_to_map(list_a)
    add_to_map(list_b)
    
    fused_sorted = sorted(scores_map.values(), key=lambda x: x['score'], reverse=True)
    return [item['doc'] for item in fused_sorted]

def rerank_contexts(query, contexts, reranker, top_k=5):
    """
    Réordonne les candidats en utilisant un Cross-Encoder.
    C'est le 'Juge' qui lit la question et le document ensemble.
    """
    if not contexts: return []
    pairs = [[query, doc['text']] for doc in contexts]
    scores = reranker.predict(pairs)
    for i, doc in enumerate(contexts):
        doc['score'] = float(scores[i])
    # Tri décroissant selon le nouveau score de pertinence
    return sorted(contexts, key=lambda x: x['score'], reverse=True)[:top_k]

## 7. Affichage et Transparence

Cette fonction est essentielle pour l'**explicabilité** (Explainability). Elle permet à l'utilisateur de vérifier l'information en affichant les "preuves" utilisées par le LLM.



**Les métadonnées affichées :**
* **Source et Chunk** : Pour retrouver le document original (ex: page 12 du PDF X).
* **Score de Confiance** : Le score final issu du *Cross-Encoder*. Plus il est élevé, plus le passage est pertinent pour la question.
* **Snippet** : Un aperçu rapide du texte pour valider le contexte d'un coup d'œil sans ouvrir le document.

In [10]:
# FONCTION D'AFFICHAGE DES SOURCES

def display_sources(contexts):
    """
    Affiche un bloc visuel avec les sources utilisées pour générer la réponse.
    Critère d'acceptation : Nom fichier, ID Chunk, Score.
    """
    if not contexts:
        return

    print("\n" + "   " + "─"*50)
    print("   📚 SOURCES UTILISÉES (Preuves)")
    print("   " + "─"*50)
    
    for i, doc in enumerate(contexts):
        # Récupération des métadonnées
        source = doc.get('source', 'Document inconnu')
        doc_id = doc.get('doc_id', '?')
        score = doc.get('score', 0.0)
        
        # Petit extrait du texte pour le contexte (optionnel mais classe)
        snippet = doc['text'][:85].replace("\n", " ") + "..."
        
        # Affichage formaté
        # On met le score en gras ou en évidence visuelle simple
        print(f"   {i+1}. 📄 {source} | Chunk #{doc_id}")
        print(f"      🎯 Confiance : {score:.4f}")
        print(f"      📝 Extrait : \"{snippet}\"")
        print("   " + "-"*20)
    print("\n")

## 8. Pipelines RAG

Nous implémentons ici **4 architectures différentes**, de la plus simple à la plus robuste. Cela permet de comparer l'impact de chaque brique technologique (Reranking, Hybrid, etc.).



### Comparatif des Stratégies :

#### 🔹 V1 : Simple RAG (Baseline)
* **Mécanisme** : Question $\rightarrow$ FAISS $\rightarrow$ LLM.
* **Avantage** : Très rapide (< 1s).
* **Inconvénient** : Manque de précision si la question ne matche pas exactement le vocabulaire des documents.

#### 🔹 V2 : Multi-Query (Expansion)
* **Mécanisme** : Le LLM génère d'abord 2 variantes de la question pour couvrir plus d'angles.
* **Intérêt** : Améliore le **Rappel (Recall)**. Utile si l'utilisateur pose une question vague.

#### 🔹 V3 : Reranking (Filtrage de Précision)
* **Mécanisme** : "Two-Stage Retrieval".
    1.  On récupère beaucoup de documents (Top-15) via FAISS.
    2.  On utilise le **Cross-Encoder** pour trier et ne garder que le Top-5 réel.
* **Intérêt** : Élimine les faux positifs. Très efficace pour des documents techniques.

#### 🔹 V4 : Hybrid Advanced RAG
C'est l'architecture la plus complète, combinant toutes les meilleures pratiques actuelles :
1.  **Expansion** (Multi-Query).
2.  **Recherche Hybride** (FAISS pour le sens + BM25 pour les mots-clés).
3.  **Fusion RRF** (Reciprocal Rank Fusion) pour unifier les scores.
4.  **Reranking** final pour la précision la plus élevée.

In [15]:
# PIPELINES RAG (Versions V1 à V4)

# ====== V1: SIMPLE RAG (Baseline) ======
def rag_v1(query, df, index, embed_model, top_k=5):
    docs = retrieve_faiss(query, df, index, embed_model, top_k)
    display_sources(docs)
    if not docs or docs[0]['score'] < THRESHOLD_SIMPLE:
        return _fallback(query, "Low FAISS Score")
    return call_llm(build_prompt(query, docs))



# ====== V2: MULTI-QUERY (Expansion de requête) ======
def rag_v2(query, df, index, embed_model, top_k=5):
    print("   [V2] Generating variations...")
    variations = [query]
    try:
        gen = call_llm_baseline(f"Generate 2 alternative questions for: {query}")
        variations += [line for line in gen.split('\n') if "?" in line][:2]
    except: pass
    
    candidates = {}
    for q in variations:
        for doc in retrieve_faiss(q, df, index, embed_model, top_k=3):
            candidates[doc['text']] = doc
            
    final_docs = list(candidates.values())[:top_k*2]
    if not final_docs: return _fallback(query)
    return call_llm(build_prompt(query, final_docs[:top_k]))






# ====== V3: RERANKING (Filtrage Avancé) ======
def rag_v3(query, df, index, embed_model, reranker, top_k=5):
    variations = [query]
    
    # Récupération large (Recall)
    candidates = {}
    for q in variations:
        for doc in retrieve_faiss(q, df, index, embed_model, top_k=15):
            candidates[doc['text']] = doc
    
    unique_candidates = list(candidates.values())
    
    # Filtrage précis (Precision)
    final_docs = rerank_contexts(query, unique_candidates, reranker, top_k=top_k)
    
    if final_docs:
        print(f"   [V3] Top Score après Rerank: {final_docs[0]['score']:.2f}")

    if not final_docs or final_docs[0]['score'] < THRESHOLD_RERANK:
        return _fallback(query, "Low Rerank Score (< 0.0)")
        
    return call_llm(build_prompt(query, final_docs))




# ====== V4: HYBRID ======
def rag_v4_hybrid(query, df, index, embed_model, bm25, reranker, top_k=5):
    print("=== [V4] Pipeline: Multi-Query -> Hybrid (FAISS+BM25) -> RRF -> Reranking ===")
    
    # 1. Expansion
    variations = [query]
    candidates = {}
    
    for q in variations:
        # A. Dense Retrieval
        res_faiss = retrieve_faiss(q, df, index, embed_model, top_k=10)
        # B. Sparse Retrieval
        res_bm25 = retrieve_bm25(q, df, bm25, top_k=10)
        # C. Fusion
        fused = reciprocal_rank_fusion(res_faiss, res_bm25)
        
        for doc in fused[:10]:
            candidates[doc['text']] = doc
            
    unique_candidates = list(candidates.values())
    print(f"   [V4] {len(unique_candidates)} candidats uniques identifiés.")

    # 2. Reranking
    final_docs = rerank_contexts(query, unique_candidates, reranker, top_k=top_k)
    
    if not final_docs: return _fallback(query)
    
    print(f"   [V4] Meilleur Score Final: {final_docs[0]['score']:.2f}")
    
    display_sources(final_docs)

    if final_docs[0]['score'] < THRESHOLD_RERANK:
         return _fallback(query, "Irrelevant Context")
    
    return call_llm(build_prompt(query, final_docs))

## 9. Démo : Comparatif  d'algorithmes (V1 vs V4)

Pour cette démonstration finale, nous n'allons pas seulement poser une question. Nous allons comparer en temps réel la **Baseline (V1)** contre notre **Architecture finale (V4)**.

In [16]:
# DEMO

def demo():
    df, index, embed_model, reranker, bm25 = load_resources()

    print("\n" + "="*60)
    print("      RAG SYSTEM : DEMO (Comparative Mode)      ")
    print("      Comparez V1 (Simple) vs V4 (Hybride Avancée)")
    print("="*60)
    
    while True:
        try:
            query = input("\nVotre question (ou 'q' pour quitter) : ").strip()
        except EOFError:
            break
            
        if query.lower() in {"q", "quit", "exit"}:
            print("Arrêt du système. Au revoir !")
            break
        
        if not query:
            continue

        print("\n" + "-"*30 + " V1. SIMPLE (FAISS) " + "-"*30)
        print(rag_v1(query, df, index, embed_model))

        print("\n" + "-"*30 + " V4. HYBRID ULTIMATE (BM25+FAISS+RERANK) " + "-"*30)
        print(rag_v4_hybrid(query, df, index, embed_model, bm25, reranker))
        
        print("\n" + "="*60)

demo()

--- Loading RAG Resources ---
Loading Reranker (Cross-Encoder)...
Building BM25 Index (Sparse Retrieval)...

      RAG SYSTEM : DEMO (Comparative Mode)      
      Comparez V1 (Simple) vs V4 (Hybride Avancée)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



------------------------------ V1. SIMPLE (FAISS) ------------------------------

   ──────────────────────────────────────────────────
   📚 SOURCES UTILISÉES (Preuves)
   ──────────────────────────────────────────────────
   1. 📄 FAISS | Chunk #1
      🎯 Confiance : 0.4239
      📝 Extrait : "2016;164:8-13. doi: 10.1016/j.drugalcdep.2016.02.044 67. Lal V, Kant S, Dewan R, Rai ..."
   --------------------
   2. 📄 FAISS | Chunk #1
      🎯 Confiance : 0.4016
      📝 Extrait : "Secondary outcomes (mor-tality and HIV transmission), which are less dependent on a s..."
   --------------------
   3. 📄 FAISS | Chunk #1
      🎯 Confiance : 0.3847
      📝 Extrait : "Zachariah (2008) Kenya n=435 [ 45]Cohort, 2005 People living with HIV receiving free ..."
   --------------------
   4. 📄 FAISS | Chunk #1
      🎯 Confiance : 0.3736
      📝 Extrait : "Comprehensive Ryan White assistance and humanimmunodeficiency virus clinical outcomes..."
   --------------------
   5. 📄 FAISS | Chunk #1
      🎯 Con

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** [RAG FAILED: Low FAISS Score] Fallback to Baseline (Connaissances générales) ***

The main symptoms of HIV (Human Immunodeficiency Virus) can vary from person to person, but here are some common ones:

1. **Painful Sores**: HIV can cause sores or ulcers on the skin, especially in the mouth, genital area, or rectum. These sores can be painful and may bleed easily.

2. **Fatigue**: People with HIV may feel extremely tired or weak, even after getting enough rest. This is because the virus is weakening the immune system.

3. **Weight Loss**: HIV can cause weight loss due to a decrease in appetite or a decrease in the body's ability to absorb nutrients.

4. **Diarrhea**: Some people with HIV may experience diarrhea, which can be caused by the virus or by other factors such as a weakened immune system.

5. **Cough**: A persistent cough can be a symptom of HIV, especially if it's caused by a virus or bacteria.

6. **Sore Throat**: A sore throat can be a symptom of HIV, especially if it's 